# Instance AI — Workflow-Builder Failure Report

Where the workflow-builder sub-agent is getting stuck, sliced by failure mode.

## Why this exists

The workflow-builder sub-agent runs autonomously inside a sandbox: it writes
TypeScript workflow code, runs `tsc`, and submits via `submit-workflow`. When
it fails, the only signal back to the user is "the agent didn't finish". This
report breaks that opaque category into named, actionable failure modes so we
can prioritize fixes against impact.

## Methodology

**Scope.** Production LangSmith project `instance-ai`, sliding 7-day window.
The unit of analysis is the **conversation thread**: a single user's session
with the agent, possibly spanning many user messages and several builder
invocations.

**Two-layer detection.**

1. **Heuristics (cheap, exhaustive).** Every workflow-builder invocation in
   the window is labelled with zero-or-more violation codes. Heuristics run
   on trace metadata only — no LLM. Thresholds are `p95 of the window ∨ a
   hard floor`, so they auto-adjust to volume but don't dip below
   sanity-checked minimums.
2. **LLM narratives (deep, sampled).** Threads flagged by the heuristics are
   stratified-sampled (≤100), and Opus 4.7 reads each thread's full
   tool-call history to produce a short failure narrative + a short
   `pattern_tag`. A second Opus pass clusters the tags into themes.

The heuristics catch *what* the agent did wrong (loop, thrash, fail). The
LLM narratives explain *why* — the specific node, error message, or contract
the agent kept tripping on.

### Heuristics in this report

| Code | Triggers when builder has… | Floor |
|---|---|---|
| `step_explosion` | ≥ N LLM calls (the loop signal) | 15 |
| `ts_error_loop` | ≥ N `execute_command` (i.e. `tsc`) calls | 5 |
| `edit_thrash` | ≥ N `edit_file` calls | 7 |
| `write_thrash` | ≥ N `write_file` calls | 4 |
| `submit_loop` | ≥ N `submit-workflow` calls | 3 |
| `latency_outlier` | ≥ N seconds wall-clock | 600 |
| `token_outlier` | ≥ N total tokens | 500k |
| `hard_failure` | run errored or `final_status=failed` | — |
| `cancelled` | `final_status=cancelled` | — |
| `builder_retry_in_thread` | thread has ≥ 2 builders, ≥ 1 failed | — |

The dynamic threshold for each metric is shown in the next section.

### LLM sampling — how 100 threads stand in for ~360

Not every flagged thread is sent through Opus (cost, time). We pick threads
by **co-occurrence cluster**: every distinct combination of violation codes
(e.g. `{step_explosion, submit_loop}`) gets representation, capped per
cluster, plus a top-severity tail. The result is breadth across failure
patterns rather than depth on a single one.

## How to read this report

| Section | Question it answers |
|---|---|
| 1. Distributions | Where do builders sit on each metric? Where are the cliffs? |
| 2. Violation counts | How many threads/builders does each heuristic flag? |
| 3. Co-occurrence | Which violations show up together? (Jaccard ≈ 1 → collapse) |
| 4. Top examples per failure mode | The worst offenders, with deep-links into LangSmith |
| 5. Per-thread drilldown | All flagged threads, sortable |
| 6. Worst offenders by metric | Outliers regardless of threshold |
| 7. **LLM-narrated themes** | The "why" — clustered by Opus from the full traces |

## Caveats to keep in mind

- **The LLM sample is stratified, not random.** Theme counts are proportional
  *within the sample*, not the full population.
- **`pattern_tag` per thread is non-deterministic.** Re-running narration may
  produce slightly different tags. The clustered themes are stable.
- **Many "failures" are user-side or infra-side**, not workflow-builder bugs:
  expired OAuth tokens during human approval, 502s from the upstream
  gateway, Cloudflare cancellations, sandbox cold-start timeouts. Treat
  themes accordingly when prioritizing fixes.


In [1]:
WINDOW = '20260430-20260507'  # set this to the window you want

# LangSmith URL parameters — used to deep-link traces. Override if your tenant differs.
LANGSMITH_TENANT_ID = 'a7b41b9d-bee0-4cf0-9786-f4600380f803'
LANGSMITH_PROJECT_ID = 'cbeb4b95-5d9c-40b4-81f5-6239a427d632'  # instance-ai
LANGSMITH_HOST = 'https://smith.langchain.com'

import json, sys
from pathlib import Path
from collections import Counter, defaultdict
from time import perf_counter

_setup_started_at = perf_counter()

def log_setup(message):
    print(f'[{perf_counter() - _setup_started_at:7.2f}s] {message}', flush=True)

log_setup('Starting trace analysis notebook setup')
log_setup(f'Working directory: {Path.cwd()}')
log_setup('Importing pandas')
import pandas as pd
log_setup('Imported pandas')
log_setup('Importing plotly')
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'vscode' if 'vscode' in pio.renderers else 'notebook_connected'
import plotly.graph_objects as go
log_setup(f'Imported plotly; renderer={pio.renderers.default}')
log_setup('Importing IPython display helpers')
from IPython.display import display, Markdown, HTML
log_setup('Imported IPython display helpers')

def find_trace_analysis_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / 'data/classified').exists():
            return base
        nested = base / 'packages/@n8n/instance-ai/scripts/trace-analysis/data/classified'
        if nested.exists():
            return nested.parent.parent
    raise FileNotFoundError('Could not find trace-analysis/data/classified from the current working directory')

log_setup('Locating trace-analysis directory')
TRACE_ANALYSIS_DIR = find_trace_analysis_dir()
log_setup(f'Trace-analysis directory: {TRACE_ANALYSIS_DIR}')
CLASSIFIED_DIR = TRACE_ANALYSIS_DIR / 'data/classified'
DATA = CLASSIFIED_DIR / f'{WINDOW}.json'
log_setup(f'Looking for classified data: {DATA}')
if not DATA.exists():
    available = ', '.join(sorted(p.stem for p in CLASSIFIED_DIR.glob('*.json'))) or 'none'
    raise FileNotFoundError(f'No classified data for WINDOW={WINDOW!r} at {DATA}. Available windows: {available}')
log_setup(f'Reading classified data ({DATA.stat().st_size / 1024 / 1024:.2f} MB)')
raw_data = DATA.read_text()
log_setup(f'Read classified data ({len(raw_data) / 1024 / 1024:.2f} MB)')
log_setup('Parsing classified JSON')
blob = json.loads(raw_data)
log_setup('Parsed classified JSON')
summary = blob['summary']
log_setup('Building builders DataFrame')
builders = pd.DataFrame(blob['builders'])
log_setup(f'Built builders DataFrame ({len(builders)} rows)')
log_setup('Building threads DataFrame')
threads = pd.DataFrame(blob['threads'])
log_setup(f'Built threads DataFrame ({len(threads)} rows)')
log_setup('Setup complete')

def trace_url(trace_id):
    return f'{LANGSMITH_HOST}/o/{LANGSMITH_TENANT_ID}/projects/p/{LANGSMITH_PROJECT_ID}/r/{trace_id}?poll=false'

def thread_url(thread_id):
    # Search runs by thread_id metadata; user lands in project filtered to the thread.
    return f'{LANGSMITH_HOST}/o/{LANGSMITH_TENANT_ID}/projects/p/{LANGSMITH_PROJECT_ID}?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22{thread_id}%5C%22))%22%7D'

log_setup('Rendering summary')
print(f'Window:    {summary["window"]}')
print(f'Builders:  {summary["totals"]["builders"]}')
print(f'Threads:   {summary["totals"]["threads"]}')
print(f'Failed:    {summary["totals"]["builders_failed"]}')
print(f'Cancelled: {summary["totals"]["builders_cancelled"]}')
print(f'\nThresholds (p95 ∨ floor):')
for k, v in summary['thresholds'].items():
    print(f'  {k:14s} = {v}')


[   0.00s] Starting trace analysis notebook setup


[   0.00s] Working directory: /Users/oleg/Projects/n8n_3/packages/@n8n/instance-ai/scripts/trace-analysis


[   0.00s] Importing pandas


[   7.16s] Imported pandas


[   7.16s] Importing plotly


...

[   8.16s] Building threads DataFrame


[   8.16s] Built threads DataFrame (261 rows)


[   8.16s] Setup complete


[   8.16s] Rendering summary


Window:    20260430-20260507
Builders:  880
Threads:   261
Failed:    164
Cancelled: 18

Thresholds (p95 ∨ floor):
  llm_calls      = 38
  exec_calls     = 7
  edit_calls     = 7
  write_calls    = 4
  submit_calls   = 5
  latency_s      = 827.641999
  tokens         = 500000


## TL;DR — top-line numbers

Pulled from the classified + narrated outputs. Refresh by re-running the
notebook against a different `WINDOW`.


In [2]:
import json
from pathlib import Path
from IPython.display import Markdown, display

narrated_path = Path('data/narrated') / f'{WINDOW}.json'
nb_blob = json.loads(narrated_path.read_text()) if narrated_path.exists() else None

lines = [
    f"- **Window:** `{summary['window']}`",
    f"- **Threads in window:** {summary['totals']['threads']:,}",
    f"- **Builder invocations:** {summary['totals']['builders']:,}",
    f"- **Hard-failed builders:** {summary['totals']['builders_failed']} "
        f"({summary['totals']['builders_failed']/max(1,summary['totals']['builders'])*100:.1f}%)",
    f"- **Cancelled builders:** {summary['totals']['builders_cancelled']} "
        f"({summary['totals']['builders_cancelled']/max(1,summary['totals']['builders'])*100:.1f}%)",
]
if nb_blob:
    lines.append(f"- **LLM-narrated sample:** {nb_blob['sample_size']} threads, {len(nb_blob.get('themes', []))} themes")

display(Markdown("\n".join(lines)))

if nb_blob and nb_blob.get('themes'):
    display(Markdown("### Top failure themes from the LLM pass"))
    th = sorted(nb_blob['themes'], key=lambda x: -x['count'])
    rows = [f"| {t['count']} | **{t['name']}** | {t['description']} |" for t in th]
    display(Markdown(
        "| Threads | Theme | What it means |\n"
        "|--------:|-------|---------------|\n"
        + "\n".join(rows)
    ))


- **Window:** `20260430-20260507`
- **Threads in window:** 261
- **Builder invocations:** 880
- **Hard-failed builders:** 164 (18.6%)
- **Cancelled builders:** 18 (2.0%)
- **LLM-narrated sample:** 95 threads, 9 themes

### Top failure themes from the LLM pass

| Threads | Theme | What it means |
|--------:|-------|---------------|
| 21 | **Mock/Placeholder Credential IDs Rejected at Submit** | Agents fabricate fake credential IDs (e.g., 'mock-openai', 'WHATSAPP_CREDENTIAL_ID') or use placeholder() on string-typed credential fields, causing tsc TS2322 errors and/or submit-workflow rejections with 'You don't have access to the credentials'. Recovery requires stripping credentials, switching to newCredential(), or running setup — often after multiple edit cycles. |
| 16 | **Submit-Loop Ignoring Blocked/Internal Error Remediation** | submit-workflow returns a generic 'workflow_save_failed / internal or service error' (or stale-workflow-id error) with explicit guidance to stop and ask the user to retry, but the agent ignores the remediation and loops through repeated rewrites and resubmissions, often exhausting budget or hitting token expiry. |
| 15 | **SDK API Misuse and tsc Error Loops** | Repeated TypeScript compile failures from misusing SDK APIs: switchCase().onCase() with string keys instead of numeric indices, splitInBatches declared via plain node() instead of factory, ifElse misuse, OutputSelector mismatches, missing toolDescription, incorrect factory config shape, etc. Often fixed only after grepping node_modules .d.ts files. |
| 14 | **Credential Setup Suspension Outliving Auth Token** | Agents call the credentials setup tool which suspends the run awaiting user input; the orchestrator's auth/session token expires before the user responds, terminating the run with 'Expired token' before any workflow is saved. Often repeats identically across retries. |
| 8 | **Write Thrash and Incremental Edit Churn** | Agents repeatedly rewrite the entire workflow file or perform many sequential one-line edits (e.g. stripping credentials node-by-node, full-file rewrites of nearly-identical content, many edit_file calls instead of one bulk overwrite) instead of making targeted bulk changes, driving step explosion and token outliers. |
| 7 | **Suspended Execution/Confirmation Token Expiry** | Workflow is built and submitted successfully but post-submit executions.run or workflows.setup suspends awaiting a 'Execute workflow?' or setup confirmation prompt; the user never responds in time and the session token expires, ending in 'Expired token' without verification. |
| 7 | **Builder Finished Without Submitting workflow.ts** | Agents write planning/chunk files under /chunks/ or alternate filenames, or run out of turns during SDK exploration, never calling submit-workflow on /src/workflow.ts — surfaced by the orchestrator as 'workflow builder finished without submitting'. |
| 6 | **Sandbox/Infrastructure Outage (Expired Sandbox Token, 502, Bad Gateway)** | Failures caused by external infrastructure: Daytona sandbox session tokens dying so every execute_command/write_file returns 'Expired token'; Cloudflare 502/origin_bad_gateway from the orchestrator API; 'failed to get runner info' on sandbox provisioning; Service Unavailable. No code defect — agents often loop retrying instead of escalating. |
| 6 | **Verify-Built-Workflow Failures Left Unresolved** | submit-workflow succeeds but verify-built-workflow / executions.run reveals real runtime issues (missing credentials, unresolved RLC documentId/sheetName placeholders, malformed Code node return shape, restricted IP/localhost calls, MySQL connection refused, IMAP timeout, OpenAI quota) that the agent dismisses as 'expected' or fails to fix, leaving a saved-but-non-executing workflow. |

## 1. Distributions

Where do builders sit on the curve? Where are the cliffs? Threshold lines are dashed.

In [3]:
metrics = [
    ('llm_calls',     'Step count (LLM calls)',          summary['thresholds']['llm_calls']),
    ('latency_s',     'Latency (s)',                     summary['thresholds']['latency_s']),
    ('tokens',        'Total tokens',                    summary['thresholds']['tokens']),
    ('exec_calls',    'mastra_workspace_execute_command', summary['thresholds']['exec_calls']),
    ('edit_calls',    'mastra_workspace_edit_file',      summary['thresholds']['edit_calls']),
    ('write_calls',   'mastra_workspace_write_file',     summary['thresholds']['write_calls']),
    ('submit_calls',  'submit-workflow',                 summary['thresholds']['submit_calls']),
]

from plotly.subplots import make_subplots
fig = make_subplots(rows=4, cols=2, subplot_titles=[m[1] for m in metrics] + [''])
for i, (col, _label, thr) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    vals = builders[col].dropna()
    fig.add_trace(go.Histogram(x=vals, nbinsx=40, marker_color='#5b8def', showlegend=False), row=row, col=c)
    fig.add_vline(x=thr, line_dash='dash', line_color='crimson', row=row, col=c)
fig.update_layout(height=900, title_text='Per-builder metric distributions (red dashed = threshold)', bargap=0.05)
fig.show()

## 2. Violation counts (per thread vs. per builder)

A single bad thread often contains multiple bad builders, so per-thread is the more honest "how many distinct user sessions had this problem" view.

In [4]:
vc_b = summary['violation_counts_per_builder']
vc_t = summary['violation_counts_per_thread']
all_codes = sorted(set(vc_b) | set(vc_t))

df_v = pd.DataFrame({
    'code': all_codes,
    'per_builder': [vc_b.get(c, 0) for c in all_codes],
    'per_thread': [vc_t.get(c, 0) for c in all_codes],
})
df_v['pct_threads'] = (df_v['per_thread'] / summary['totals']['threads'] * 100).round(1)
df_v = df_v.sort_values('per_thread', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=df_v['code'], x=df_v['per_thread'], orientation='h', name='threads', marker_color='#ef5b5b'))
fig.add_trace(go.Bar(y=df_v['code'], x=df_v['per_builder'], orientation='h', name='builders', marker_color='#5b8def', opacity=0.6))
fig.update_layout(barmode='overlay', height=420, title='Violations: distinct threads vs total builder events')
fig.show()

display(df_v.sort_values('per_thread', ascending=False).reset_index(drop=True))

,code,per_builder,per_thread,pct_threads
0,builder_retry_in_thread,0,74,28.4
1,hard_failure,164,72,27.6
2,submit_loop,44,37,14.2
3,latency_outlier,43,34,13.0
4,ts_error_loop,54,33,12.6
5,write_thrash,34,32,12.3
6,token_outlier,38,28,10.7
7,edit_thrash,30,23,8.8
8,step_explosion,45,22,8.4
9,cancelled,18,16,6.1


## 3. Co-occurrence matrix

Which violations show up together? Cell `(i, j)` = number of threads that triggered both `i` and `j`. Diagonal = total threads with `i`. Strong off-diagonal blocks suggest correlated codes that we may want to collapse into one.

In [5]:
all_codes_t = sorted(vc_t.keys())
co = pd.DataFrame(0, index=all_codes_t, columns=all_codes_t, dtype=int)
for _, row in threads.iterrows():
    codes = row['violation_codes'] or []
    for a in codes:
        for b in codes:
            if a in co.index and b in co.columns:
                co.loc[a, b] += 1

fig = px.imshow(
    co.values,
    x=co.columns, y=co.index,
    color_continuous_scale='Reds',
    text_auto=True,
    aspect='auto',
    title='Per-thread violation co-occurrence (cell = #threads with both)'
)
fig.update_layout(height=520)
fig.show()

# Jaccard similarity between codes — easier to spot correlated pairs.
import numpy as np
diag = np.diag(co.values).astype(float)
with np.errstate(divide='ignore', invalid='ignore'):
    union = diag[:, None] + diag[None, :] - co.values
    jac = np.where(union > 0, co.values / union, 0.0)
np.fill_diagonal(jac, 0.0)
fig2 = px.imshow(jac, x=co.columns, y=co.index, color_continuous_scale='Blues', text_auto='.2f', aspect='auto',
                title='Jaccard similarity (1.0 = always co-occur)')
fig2.update_layout(height=520)
fig2.show()

## 4. Top examples per failure mode

For each heuristic, the 10 worst builders by the underlying metric. Click the trace_id to open in LangSmith; the thread link filters the project to all runs in that conversation.

In [6]:
PANEL_BY_CODE = {
    'step_explosion':  'llm_calls',
    'ts_error_loop':   'exec_calls',
    'edit_thrash':     'edit_calls',
    'write_thrash':    'write_calls',
    'submit_loop':     'submit_calls',
    'latency_outlier': 'latency_s',
    'token_outlier':   'tokens',
    'hard_failure':    None,
    'cancelled':       None,
}

builders['violation_codes'] = builders['violations'].apply(
    lambda vs: [v['code'] for v in (vs or [])]
)

for code, sort_col in PANEL_BY_CODE.items():
    sub = builders[builders['violation_codes'].apply(lambda xs: code in xs)].copy()
    if sub.empty:
        continue
    if sort_col:
        sub = sub.sort_values(sort_col, ascending=False)
    sub = sub.head(10)
    rows = []
    for _, b in sub.iterrows():
        rows.append({
            'when': (b['start_time'] or '')[:16].replace('T', ' '),
            'thread': f'<a href="{thread_url(b["thread_id"])}" target="_blank">{(b["thread_id"] or "?")[:8]}</a>' if b['thread_id'] else '?',
            'trace':  f'<a href="{trace_url(b["trace_id"])}" target="_blank">{b["trace_id"][:8]}</a>',
            'llm':    int(b['llm_calls']),
            'exec':   int(b['exec_calls']),
            'edit':   int(b['edit_calls']),
            'write':  int(b['write_calls']),
            'submit': int(b['submit_calls']),
            'latency_s': round(b['latency_s'], 0) if b['latency_s'] else None,
            'tokens': int(b['tokens'] or 0),
            'status': f'{b["status"]} / {b["final_status"]}',
        })
    df = pd.DataFrame(rows)
    display(Markdown(f'### `{code}` — top {len(df)} (sorted by `{sort_col or "recency"}`)'))
    display(HTML(df.to_html(escape=False, index=False)))


### `step_explosion` — top 10 (sorted by `llm_calls`)

when,thread,trace,llm,exec,edit,write,submit,latency_s,tokens,status
2026-05-03 17:06,a2035bfe,019deece,360,1,2,0,1,443.0,141220,success / completed
2026-05-03 04:29,a2035bfe,019dec18,286,1,2,0,1,243.0,179266,success / completed
2026-04-30 11:30,e408d5cf,019dde27,192,1,0,0,1,388.0,296400,success / completed
2026-04-30 14:02,e408d5cf,019ddeb2,146,2,8,0,2,654.0,191627,error / failed
2026-04-30 14:13,e408d5cf,019ddebc,139,0,1,0,2,478.0,380561,success / completed
2026-05-03 04:49,a2035bfe,019dec2b,137,2,4,1,1,323.0,302487,success / completed
2026-05-03 18:01,a2035bfe,019def00,130,1,2,0,1,228.0,68736,success / completed
2026-05-02 20:41,a2035bfe,019dea6c,115,5,3,1,4,370.0,112761,success / completed
2026-04-30 09:19,e408d5cf,019dddaf,98,0,1,0,1,374.0,228201,success / completed
2026-04-30 17:03,e408d5cf,019ddf58,92,4,1,0,1,351.0,223829,success / completed


### `ts_error_loop` — top 10 (sorted by `exec_calls`)

when,thread,trace,llm,exec,edit,write,submit,latency_s,tokens,status
2026-05-07 06:52,2b8a0aeb,019e0135,60,60,0,0,0,193.0,11950,error / failed
2026-05-07 03:15,2b8a0aeb,019e006f,60,59,0,0,0,164.0,1370252,error / failed
2026-05-07 06:46,2b8a0aeb,019e0130,60,57,0,1,1,262.0,40172,error / failed
2026-05-01 21:02,62949ba2,019de559,58,54,0,0,1,588.0,0,success / completed
2026-05-01 20:39,62949ba2,019de544,52,48,2,0,0,616.0,0,error / failed
2026-05-01 20:50,62949ba2,019de54e,53,37,1,0,0,659.0,0,error / failed
2026-05-07 06:29,3c92a183,019e0120,63,35,0,2,9,1038.0,238283,success / completed
2026-05-01 19:27,62949ba2,019de502,35,27,1,1,1,339.0,64811,success / completed
2026-05-05 08:22,5c8f38fa,019df73b,34,20,4,0,1,NaN,103108,pending / nan
2026-05-02 17:29,e408d5cf,019de9bd,45,20,4,0,2,595.0,13071,success / completed


### `edit_thrash` — top 10 (sorted by `edit_calls`)

...

when,thread,trace,llm,exec,edit,write,submit,latency_s,tokens,status
2026-04-30 20:52,6c1c0568,019de02a,45,1,12,3,10,522.0,2954309,success / completed
2026-05-06 11:53,f7e43a66,019dfd22,40,7,8,4,8,551.0,1748744,success / completed
2026-04-30 11:44,b1d73388,019dde33,28,8,1,4,5,NaN,1715236,pending / nan
2026-05-07 04:22,2b8a0aeb,019e00ac,40,6,2,5,10,554.0,1665813,success / completed
2026-05-04 17:27,4fb6b031,019df407,30,15,6,1,1,NaN,1385106,pending / nan
2026-05-07 03:15,2b8a0aeb,019e006f,60,59,0,0,0,164.0,1370252,error / failed
2026-05-03 04:35,35f574e1,019dec1f,29,7,6,1,1,399.0,1306392,success / completed
2026-05-04 12:29,5e3e9516,019df2f6,26,4,11,1,2,NaN,1248278,pending / nan
2026-05-01 18:55,62949ba2,019de4e5,36,16,1,2,1,579.0,1210172,success / completed
2026-05-04 09:55,812a40f5,019df26a,38,1,3,6,9,492.0,1166428,success / completed


### `hard_failure` — top 10 (sorted by `recency`)

when,thread,trace,llm,exec,edit,write,submit,latency_s,tokens,status
2026-05-07 08:20,2b8a0aeb,019e0186,10,1,0,3,1,72.0,431007,error / failed
2026-05-07 08:17,2b8a0aeb,019e0183,10,0,1,2,0,96.0,356865,error / failed
2026-05-07 08:11,2b8a0aeb,019e017e,13,2,0,2,1,104.0,351251,error / failed
2026-05-07 08:01,2b8a0aeb,019e0174,9,2,0,2,1,109.0,171217,error / failed
2026-05-07 07:50,2b8a0aeb,019e016a,12,4,0,2,1,105.0,93162,error / failed
2026-05-07 07:23,2b8a0aeb,019e0152,4,1,0,1,1,35.0,6201,error / failed
2026-05-07 07:18,2b8a0aeb,019e014d,5,2,0,1,0,29.0,8898,error / failed
2026-05-07 07:17,2b8a0aeb,019e014c,15,6,0,2,0,81.0,20878,error / failed
2026-05-07 06:52,2b8a0aeb,019e0135,60,60,0,0,0,193.0,11950,error / failed
2026-05-07 06:46,2b8a0aeb,019e0130,60,57,0,1,1,262.0,40172,error / failed


### `cancelled` — top 10 (sorted by `recency`)

when,thread,trace,llm,exec,edit,write,submit,latency_s,tokens,status
2026-05-07 06:47,3c92a183,019e0130,3,0,0,0,0,19.0,7094,success / cancelled
2026-05-07 03:20,2b8a0aeb,019e0073,14,13,0,0,0,33.0,710606,success / cancelled
2026-05-06 20:16,bf5a0c0b,019dfeef,1,0,0,0,0,2.0,0,success / cancelled
2026-05-06 09:36,faf4abae,019dfca5,4,0,0,0,0,85.0,20100,success / cancelled
2026-05-06 08:06,34518acc,019dfc52,11,0,0,0,0,87.0,2401,success / cancelled
2026-05-05 00:28,6035e1fd,019df589,21,3,6,1,2,1367.0,651966,success / cancelled
2026-05-04 03:45,a2035bfe,019df117,48,0,2,0,1,221.0,102966,success / cancelled
2026-05-04 03:21,c4f11df0,019df101,9,2,1,1,1,545.0,39814,success / cancelled
2026-05-03 18:04,702df03c,019def03,16,3,2,1,2,348.0,551466,success / cancelled
2026-05-03 10:02,6cbefec4,019ded49,1,0,0,0,0,4.0,0,success / cancelled


## 5. Per-thread drilldown

All threads sorted by violation count, then total tokens. Look here when a specific thread keeps showing up across multiple panels.

In [7]:
drill = threads.copy()
drill['violation_count'] = drill['violation_codes'].apply(len)
drill['violations'] = drill['violation_codes'].apply(lambda xs: ', '.join(xs))
drill['link'] = drill['thread_id'].apply(lambda t: f'<a href="{thread_url(t)}" target="_blank">{t[:8]}</a>')
drill['first'] = drill['first_start'].str[:16].str.replace('T', ' ')
drill = drill[['link', 'first', 'builder_count', 'fail_count', 'total_tokens', 'violation_count', 'violations']]
drill = drill.rename(columns={'link': 'thread'})
drill = drill.sort_values(['violation_count', 'total_tokens'], ascending=[False, False])

display(Markdown(f'**{len(drill)} threads, top 30 shown**'))
display(HTML(drill.head(30).to_html(escape=False, index=False)))

**261 threads, top 30 shown**

thread,first,builder_count,fail_count,total_tokens,violation_count,violations
2b8a0aeb,2026-05-07 03:02,35,19,9708482,10,"builder_retry_in_thread, cancelled, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, token_outlier, ts_error_loop, write_thrash"
e408d5cf,2026-04-30 09:19,57,10,4517463,8,"builder_retry_in_thread, cancelled, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, ts_error_loop"
812a40f5,2026-05-04 09:55,10,1,2318406,8,"builder_retry_in_thread, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, token_outlier, write_thrash"
d36354ba,2026-05-06 15:12,2,1,735876,8,"builder_retry_in_thread, hard_failure, latency_outlier, step_explosion, submit_loop, token_outlier, ts_error_loop, write_thrash"
5d441e6e,2026-05-06 15:03,5,4,253518,8,"builder_retry_in_thread, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, ts_error_loop, write_thrash"
6f10d39b,2026-05-06 17:56,3,1,243710,8,"builder_retry_in_thread, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, ts_error_loop, write_thrash"
541c085b,2026-05-04 16:17,8,1,1177561,7,"builder_retry_in_thread, hard_failure, latency_outlier, submit_loop, token_outlier, ts_error_loop, write_thrash"
098c5695,2026-05-05 11:43,1,1,638468,7,"hard_failure, latency_outlier, step_explosion, submit_loop, token_outlier, ts_error_loop, write_thrash"
3bf06444,2026-05-06 09:10,2,1,358307,7,"builder_retry_in_thread, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, write_thrash"
d33e7028,2026-05-02 13:47,3,1,156073,7,"builder_retry_in_thread, edit_thrash, hard_failure, latency_outlier, step_explosion, submit_loop, write_thrash"


## 6. Worst offenders by metric

Top builders for each individual metric, regardless of whether they crossed a threshold. Useful for spotting near-misses or extreme outliers.

In [8]:
def top_by(col, n=10):
    sub = builders.sort_values(col, ascending=False).head(n).copy()
    sub['trace'] = sub['trace_id'].apply(lambda t: f'<a href="{trace_url(t)}" target="_blank">{t[:8]}</a>')
    sub['thread'] = sub['thread_id'].apply(lambda t: f'<a href="{thread_url(t)}" target="_blank">{(t or "?")[:8]}</a>' if t else '?')
    sub['violations'] = sub['violation_codes'].apply(lambda xs: ', '.join(xs))
    keep = ['thread', 'trace', col, 'llm_calls', 'exec_calls', 'edit_calls', 'write_calls', 'submit_calls', 'latency_s', 'tokens', 'status', 'final_status', 'violations']
    return sub[[c for c in keep if c in sub.columns]]

for col in ['llm_calls', 'latency_s', 'tokens', 'exec_calls', 'edit_calls', 'write_calls', 'submit_calls']:
    display(Markdown(f'### Top by `{col}`'))
    display(HTML(top_by(col).to_html(escape=False, index=False)))

### Top by `llm_calls`

thread,trace,llm_calls,llm_calls,exec_calls,edit_calls,write_calls,submit_calls,latency_s,tokens,status,final_status,violations
a2035bfe,019deece,360,360,1,2,0,1,442.792999,141220,success,completed,step_explosion
a2035bfe,019dec18,286,286,1,2,0,1,242.926999,179266,success,completed,step_explosion
e408d5cf,019dde27,192,192,1,0,0,1,388.166999,296400,success,completed,step_explosion
e408d5cf,019ddeb2,146,146,2,8,0,2,653.690999,191627,error,failed,"step_explosion, edit_thrash, hard_failure"
e408d5cf,019ddebc,139,139,0,1,0,2,477.660999,380561,success,completed,step_explosion
a2035bfe,019dec2b,137,137,2,4,1,1,323.346999,302487,success,completed,step_explosion
a2035bfe,019def00,130,130,1,2,0,1,228.058999,68736,success,completed,step_explosion
a2035bfe,019dea6c,115,115,5,3,1,4,369.775999,112761,success,completed,step_explosion
e408d5cf,019dddaf,98,98,0,1,0,1,373.506999,228201,success,completed,step_explosion
e408d5cf,019ddf58,92,92,4,1,0,1,350.504999,223829,success,completed,step_explosion


### Top by `latency_s`

thread,trace,latency_s,llm_calls,exec_calls,edit_calls,write_calls,submit_calls,latency_s,tokens,status,final_status,violations
8b690188,019df351,5354.757999,5,0,0,0,0,5354.757999,28116,error,failed,"latency_outlier, hard_failure"
3bf06444,019dfc8d,3073.253999,9,0,0,0,0,3073.253999,117542,error,failed,"latency_outlier, hard_failure"
64ccb2d4,019de2df,2757.493999,12,1,2,1,3,2757.493999,88842,error,failed,"latency_outlier, hard_failure"
d3403b31,019de11a,1543.317999,6,0,0,0,0,1543.317999,34652,error,failed,"latency_outlier, hard_failure"
327d43b5,019ddea2,1493.613999,10,1,0,2,2,1493.613999,56550,error,failed,"latency_outlier, hard_failure"
f048925b,019df2a3,1446.260999,8,1,0,1,1,1446.260999,127262,error,failed,"latency_outlier, hard_failure"
608907d2,019ddf1c,1442.630999,8,1,0,1,1,1442.630999,171252,error,failed,"latency_outlier, hard_failure"
6035e1fd,019df589,1367.475999,21,3,6,1,2,1367.475999,651966,success,cancelled,"latency_outlier, token_outlier, cancelled"
cc4410fd,019dea7c,1360.772999,2,0,0,0,0,1360.772999,216,error,failed,"latency_outlier, hard_failure"
60536c10,019dddd5,1360.107999,6,1,0,1,1,1360.107999,10382,error,failed,"latency_outlier, hard_failure"


### Top by `tokens`

...

thread,trace,edit_calls,llm_calls,exec_calls,edit_calls,write_calls,submit_calls,latency_s,tokens,status,final_status,violations
d1302530,019dde52,21,42,2,21,2,1,NaN,0,pending,NaN,"step_explosion, edit_thrash"
d33e7028,019de8f1,16,59,6,16,4,9,531.103999,112452,success,completed,"step_explosion, edit_thrash, write_thrash, submit_loop"
d33e7028,019de8ff,13,38,6,13,1,4,NaN,0,pending,NaN,"step_explosion, edit_thrash"
d8f79dd9,019dec9a,13,37,7,13,1,3,NaN,0,pending,NaN,"ts_error_loop, edit_thrash"
3f221af9,019deea3,12,29,4,12,1,5,304.494999,969993,success,completed,"edit_thrash, submit_loop, token_outlier"
6c1c0568,019de02a,12,45,1,12,3,10,522.390999,2954309,success,completed,"step_explosion, edit_thrash, submit_loop, token_outlier"
6f10d39b,019dfe6f,12,60,16,12,2,11,517.249999,124217,success,completed,"step_explosion, ts_error_loop, edit_thrash, submit_loop"
5e3e9516,019df2f6,11,26,4,11,1,2,NaN,1248278,pending,NaN,"edit_thrash, token_outlier"
2b8a0aeb,019e00ba,11,25,1,11,0,2,NaN,408984,pending,NaN,edit_thrash
30fd6c01,019de97f,10,26,3,10,1,2,234.234999,89607,success,completed,edit_thrash


### Top by `write_calls`

thread,trace,write_calls,llm_calls,exec_calls,edit_calls,write_calls,submit_calls,latency_s,tokens,status,final_status,violations
e39daff0,019df124,8,35,3,1,8,6,434.250999,77530,success,completed,"write_thrash, submit_loop"
541c085b,019df3ff,8,28,8,6,8,4,440.046999,786965,success,completed,"ts_error_loop, write_thrash, token_outlier"
fedfb2c4,019dfc2c,8,43,6,2,8,13,520.716999,98707,success,completed,"step_explosion, write_thrash, submit_loop"
8a8d0d1c,019dff28,8,44,6,5,8,11,782.564999,0,error,failed,"step_explosion, write_thrash, submit_loop, hard_failure"
8ed19f53,019de360,7,33,3,5,7,12,NaN,161318,pending,NaN,"write_thrash, submit_loop"
b0208b8c,019dfb60,7,46,8,5,7,10,NaN,0,pending,NaN,"step_explosion, ts_error_loop, write_thrash, submit_loop"
3bf06444,019dfcbc,6,43,5,10,6,7,NaN,240765,pending,NaN,"step_explosion, edit_thrash, write_thrash, submit_loop"
812a40f5,019df26a,6,38,1,3,6,9,491.758999,1166428,success,completed,"step_explosion, write_thrash, submit_loop, token_outlier"
d36354ba,019dfdea,6,44,9,5,6,14,NaN,0,pending,NaN,"step_explosion, ts_error_loop, write_thrash, submit_loop"
0e0bcab0,019df33f,5,15,2,1,5,1,NaN,221356,pending,NaN,write_thrash


### Top by `submit_calls`

thread,trace,submit_calls,llm_calls,exec_calls,edit_calls,write_calls,submit_calls,latency_s,tokens,status,final_status,violations
d36354ba,019dfdea,14,44,9,5,6,14,NaN,0,pending,NaN,"step_explosion, ts_error_loop, write_thrash, submit_loop"
8e52e969,019dfdfb,14,58,8,10,4,14,387.917999,107890,success,completed,"step_explosion, ts_error_loop, edit_thrash, write_thrash, submit_loop"
9442f0f7,019df8f7,14,46,3,8,3,14,1106.000999,205424,error,failed,"step_explosion, edit_thrash, submit_loop, latency_outlier, hard_failure"
fedfb2c4,019dfc2c,13,43,6,2,8,13,520.716999,98707,success,completed,"step_explosion, write_thrash, submit_loop"
8ed19f53,019de360,12,33,3,5,7,12,NaN,161318,pending,NaN,"write_thrash, submit_loop"
6f10d39b,019dfe6f,11,60,16,12,2,11,517.249999,124217,success,completed,"step_explosion, ts_error_loop, edit_thrash, submit_loop"
8a8d0d1c,019dff28,11,44,6,5,8,11,782.564999,0,error,failed,"step_explosion, write_thrash, submit_loop, hard_failure"
5d441e6e,019dfde8,10,34,5,5,5,10,312.969999,60417,success,completed,"write_thrash, submit_loop"
b0208b8c,019dfb60,10,46,8,5,7,10,NaN,0,pending,NaN,"step_explosion, ts_error_loop, write_thrash, submit_loop"
6c1c0568,019de02a,10,45,1,12,3,10,522.390999,2954309,success,completed,"step_explosion, edit_thrash, submit_loop, token_outlier"


## 7. LLM-narrated failure clusters

Output of `narrate.py` — Opus 4.7 reads each sampled thread's full I/O,
produces a structured failure narrative, then a second pass clusters
the narratives into themes. Loads `data/narrated/<window>.json` if it
exists; skip this section otherwise.


In [9]:
import json
from pathlib import Path
from IPython.display import display, Markdown, HTML

narrated_path = Path('data/narrated') / f'{WINDOW}.json'
if not narrated_path.exists():
    display(Markdown(f"_No narrated data at `{narrated_path}` — run `./narrate.py --window {WINDOW}` to generate._"))
else:
    nb_blob = json.loads(narrated_path.read_text())
    nar_by_thread = {n['thread_id']: n for n in nb_blob['narratives']}
    themes = nb_blob.get('themes', [])

    display(Markdown(
        f"**Sample:** {nb_blob['sample_size']} threads &nbsp;|&nbsp; "
        f"**Model:** {nb_blob['model']} &nbsp;|&nbsp; "
        f"**Tokens:** {nb_blob['tokens']['input']:,} in / {nb_blob['tokens']['output']:,} out"
    ))

    # Theme overview chart
    if themes:
        import plotly.express as px
        df_themes = pd.DataFrame([
            {'theme': t['name'], 'count': t['count']} for t in themes
        ]).sort_values('count', ascending=True)
        fig = px.bar(df_themes, x='count', y='theme', orientation='h',
                     title=f"Themes across {nb_blob['sample_size']} sampled threads",
                     text='count')
        fig.update_layout(height=max(360, 36 * len(themes)))
        fig.show()

    # Per-theme drilldown
    for t in sorted(themes, key=lambda x: -x['count']):
        ex_links = []
        for tid in t.get('example_thread_ids', [])[:5]:
            n = nar_by_thread.get(tid)
            tag = n['narrative']['pattern_tag'] if n else '?'
            ex_links.append(f'<a href="{thread_url(tid)}" target="_blank"><code>{tid[:8]}</code></a> <em>{tag}</em>')
        display(Markdown(
            f"### {t['name']} &nbsp;<small>({t['count']} threads)</small>\n\n"
            f"{t['description']}\n\n"
            f"**Pattern tags:** " + ', '.join(f'`{tg}`' for tg in t.get('pattern_tags', [])) + "\n\n"
            f"**Examples:** " + ' &nbsp;·&nbsp; '.join(ex_links)
        ))


**Sample:** 95 threads &nbsp;|&nbsp; **Model:** claude-opus-4-7 &nbsp;|&nbsp; **Tokens:** 1,850,237 in / 61,992 out

### Mock/Placeholder Credential IDs Rejected at Submit &nbsp;<small>(21 threads)</small>

Agents fabricate fake credential IDs (e.g., 'mock-openai', 'WHATSAPP_CREDENTIAL_ID') or use placeholder() on string-typed credential fields, causing tsc TS2322 errors and/or submit-workflow rejections with 'You don't have access to the credentials'. Recovery requires stripping credentials, switching to newCredential(), or running setup — often after multiple edit cycles.

**Pattern tags:** `repeated-full-file-rewrites`, `mock-credential-id-rejected`, `mocked-credential-id-rejected`, `mock-credential-ids-before-setup`, `credential-placeholder-shape-thrash`, `placeholder-on-string-credential-fields`, `credentials-in-set-node`, `credential-mock-publish-loop`, `credential-placeholder-thrash`, `fake-credential-ids-then-strip`, `fabricated-mock-credentials`, `placeholder-credential-id-rejected`, `placeholder-credential-id-loop`, `fake-mocked-credential-ids`, `deferred-credentials-submit-failure`, `missing-credentials-mock-discovery`, `hardcoded-placeholder-credential-ids`, `mock-credential-ids-rejected`, `hardcoded-credential-query-param`, `credential-skipped-then-fake-id`, `missing-submit-and-credential-placeholder-misuse`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%220e0bcab0-2ab3-4f87-8a66-835dd453b7a8%5C%22))%22%7D" target="_blank"><code>0e0bcab0</code></a> <em>repeated-full-file-rewrites</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2208fc96e1-cd0b-457c-8df5-2879f8c159bd%5C%22))%22%7D" target="_blank"><code>08fc96e1</code></a> <em>mocked-credential-id-rejected</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2230fd6c01-4e69-4b61-8842-7b4d9566de96%5C%22))%22%7D" target="_blank"><code>30fd6c01</code></a> <em>placeholder-on-string-credential-fields</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2256f6ccee-84d8-479b-a379-aed091cf6071%5C%22))%22%7D" target="_blank"><code>56f6ccee</code></a> <em>credential-placeholder-thrash</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%226f10d39b-6011-45f8-870d-36a0edeb6769%5C%22))%22%7D" target="_blank"><code>6f10d39b</code></a> <em>hardcoded-placeholder-credential-ids</em>

### Submit-Loop Ignoring Blocked/Internal Error Remediation &nbsp;<small>(16 threads)</small>

submit-workflow returns a generic 'workflow_save_failed / internal or service error' (or stale-workflow-id error) with explicit guidance to stop and ask the user to retry, but the agent ignores the remediation and loops through repeated rewrites and resubmissions, often exhausting budget or hitting token expiry.

**Pattern tags:** `submit-loop-on-internal-error`, `submit-loop-on-credential-access-error`, `submit-loop-on-instance-error`, `ignored-blocked-remediation-submit-loop`, `opaque-save-failure-retry-loop`, `submit-workflow-internal-error-loop`, `submit-loop-ignoring-blocked-remediation`, `stale-workflow-id-submit-loop`, `submit-loop-after-blocked`, `credential-access-denied-submit-loop`, `credential-not-shared-submit-loop`, `credential-access-not-shared`, `missing-credentials-submit-loop`, `submit-fix-loop-no-credentials`, `submit-loop-on-archived-workflow`, `archived-workflow-submit-loop`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22098c5695-cfe9-4e2f-beb7-9d1addc02896%5C%22))%22%7D" target="_blank"><code>098c5695</code></a> <em>submit-loop-on-internal-error</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%229442f0f7-ad46-477a-b328-8442bb653ed9%5C%22))%22%7D" target="_blank"><code>9442f0f7</code></a> <em>submit-loop-on-credential-access-error</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%229e48435c-50c0-45f1-903b-37e22c7056f2%5C%22))%22%7D" target="_blank"><code>9e48435c</code></a> <em>ignored-blocked-remediation-submit-loop</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22b3e71f72-0f66-48fb-9f22-3de71535bca6%5C%22))%22%7D" target="_blank"><code>b3e71f72</code></a> <em>submit-loop-on-instance-error</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%228a8d0d1c-fa58-4e36-8898-c2ea253d02a3%5C%22))%22%7D" target="_blank"><code>8a8d0d1c</code></a> <em>submit-loop-ignoring-blocked-remediation</em>

### SDK API Misuse and tsc Error Loops &nbsp;<small>(15 threads)</small>

Repeated TypeScript compile failures from misusing SDK APIs: switchCase().onCase() with string keys instead of numeric indices, splitInBatches declared via plain node() instead of factory, ifElse misuse, OutputSelector mismatches, missing toolDescription, incorrect factory config shape, etc. Often fixed only after grepping node_modules .d.ts files.

**Pattern tags:** `switch-oncase-type-mismatch`, `ifelse-builder-misuse`, `vector-store-as-tool-confusion`, `sdk-api-spelunking-timeout`, `sdk-spelunking-no-edit`, `wait-node-options-loop`, `http-raw-body-specifybody-conflict`, `oversized-monolithic-workflow`, `patch-old-str-mismatch`, `template-literal-regex-escaping`, `embedded-jscode-escape-mismatch`, `multi-submit-validation-thrash`, `aggregate-include-misconfig-and-apify-auth-thrash`, `recurring-slack-discriminator-resubmit`, `debug-loop-external-api`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22d7cbe2e6-ab35-4c0b-babd-88e661b42891%5C%22))%22%7D" target="_blank"><code>d7cbe2e6</code></a> <em>switch-oncase-type-mismatch</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22cf30d6c6-2ce4-4df4-97e4-55e040d2ee79%5C%22))%22%7D" target="_blank"><code>cf30d6c6</code></a> <em>ifelse-builder-misuse</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%228e52e969-f927-4e6d-b6c8-7a723896b93e%5C%22))%22%7D" target="_blank"><code>8e52e969</code></a> <em>vector-store-as-tool-confusion</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%222d5ce6c5-97cc-4c56-9d8e-a9a588ab07d1%5C%22))%22%7D" target="_blank"><code>2d5ce6c5</code></a> <em>template-literal-regex-escaping</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%220375686f-fcbe-4766-a120-c100c296b619%5C%22))%22%7D" target="_blank"><code>0375686f</code></a> <em>http-raw-body-specifybody-conflict</em>

### Credential Setup Suspension Outliving Auth Token &nbsp;<small>(14 threads)</small>

Agents call the credentials setup tool which suspends the run awaiting user input; the orchestrator's auth/session token expires before the user responds, terminating the run with 'Expired token' before any workflow is saved. Often repeats identically across retries.

**Pattern tags:** `credential-setup-suspend-timeout`, `credentials-setup-after-submit`, `credential-setup-token-expiry`, `credential-suspend-token-expiry`, `credential-setup-instead-of-mock`, `credentials-requested-after-submit`, `expired-token-during-credential-setup`, `credential-flow-suspend-and-set-field-thrash`, `credential-mock-ignored-suspend-expiry`, `deferred-credentials-token-expiry`, `credential-setup-suspend-expired`, `credential-setup-suspend-expiry`, `data-table-suspend-token-expiry`, `cancelled-during-credential-suspension`, `faf4abae-style`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2223d500c2-f7de-47bd-9b3a-3d3dcd3224f2%5C%22))%22%7D" target="_blank"><code>23d500c2</code></a> <em>credential-setup-suspend-timeout</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22608907d2-0edf-4547-bd6a-d988ce1d40c5%5C%22))%22%7D" target="_blank"><code>608907d2</code></a> <em>credentials-setup-after-submit</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%220d317b56-5e71-45e8-9c8d-cf76a907add2%5C%22))%22%7D" target="_blank"><code>0d317b56</code></a> <em>credential-setup-token-expiry</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%228b690188-765e-4340-a997-f82b81f22559%5C%22))%22%7D" target="_blank"><code>8b690188</code></a> <em>credential-setup-suspend-expired</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22a08e72a2-0866-4847-b068-521e867e4f69%5C%22))%22%7D" target="_blank"><code>a08e72a2</code></a> <em>credential-setup-suspend-expiry</em>

### Write Thrash and Incremental Edit Churn &nbsp;<small>(8 threads)</small>

Agents repeatedly rewrite the entire workflow file or perform many sequential one-line edits (e.g. stripping credentials node-by-node, full-file rewrites of nearly-identical content, many edit_file calls instead of one bulk overwrite) instead of making targeted bulk changes, driving step explosion and token outliers.

**Pattern tags:** `repeated-full-file-rewrites`, `incremental-edit-thrash-rewrite`, `submit-run-rewrite-loop`, `iterative-refinement-submit-loop`, `webhook-payload-shape-thrash`, `notion-databaseid-thrash`, `supabase-ddl-from-n8n-thrash`, `credential-access-and-data-shape-loop`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22d1302530-715d-4498-9962-863ccc49120e%5C%22))%22%7D" target="_blank"><code>d1302530</code></a> <em>incremental-edit-thrash-rewrite</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22aa575248-01a0-4301-958e-b4d463dd94ab%5C%22))%22%7D" target="_blank"><code>aa575248</code></a> <em>submit-run-rewrite-loop</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22a2035bfe-b453-4d25-a917-caaefa64105c%5C%22))%22%7D" target="_blank"><code>a2035bfe</code></a> <em>iterative-refinement-submit-loop</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22bba6a9c2-749d-4807-a5e1-96e7cf5b69b9%5C%22))%22%7D" target="_blank"><code>bba6a9c2</code></a> <em>webhook-payload-shape-thrash</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22e39daff0-81ad-4893-b458-77a2a42900a7%5C%22))%22%7D" target="_blank"><code>e39daff0</code></a> <em>supabase-ddl-from-n8n-thrash</em>

### Suspended Execution/Confirmation Token Expiry &nbsp;<small>(7 threads)</small>

Workflow is built and submitted successfully but post-submit executions.run or workflows.setup suspends awaiting a 'Execute workflow?' or setup confirmation prompt; the user never responds in time and the session token expires, ending in 'Expired token' without verification.

**Pattern tags:** `token-expired-during-execution`, `expired-token-and-splitinbatches-loop`, `suspended-execution-token-expiry`, `execution-suspend-token-expiry`, `verify-loop-token-expiry`, `wrong-output-path-and-token-expiry`, `notion-databaseid-thrash`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2268b953c9-903f-4db5-a2eb-2f011ad0c26b%5C%22))%22%7D" target="_blank"><code>68b953c9</code></a> <em>token-expired-during-execution</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2260536c10-bd30-4575-9a00-96124007c999%5C%22))%22%7D" target="_blank"><code>60536c10</code></a> <em>execution-suspend-token-expiry</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22e408d5cf-ba61-453b-89c9-27aa839bbb38%5C%22))%22%7D" target="_blank"><code>e408d5cf</code></a> <em>suspended-execution-token-expiry</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%224fb6b031-405a-48a8-93fe-5dcca87459f5%5C%22))%22%7D" target="_blank"><code>4fb6b031</code></a> <em>expired-token-and-splitinbatches-loop</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%2299aaa802-ef0e-40c9-972e-2d91216c7e23%5C%22))%22%7D" target="_blank"><code>99aaa802</code></a> <em>verify-loop-token-expiry</em>

### Builder Finished Without Submitting workflow.ts &nbsp;<small>(7 threads)</small>

Agents write planning/chunk files under /chunks/ or alternate filenames, or run out of turns during SDK exploration, never calling submit-workflow on /src/workflow.ts — surfaced by the orchestrator as 'workflow builder finished without submitting'.

**Pattern tags:** `finished-without-submitting`, `chunked-planning-no-submit`, `missing-submit-and-credential-placeholder-misuse`, `sdk-spelunking-no-edit`, `fragmented-retry-no-submit`, `scope-creep-credential-thrash`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22b99f6ace-ccbc-4c2b-8f96-016b730d2118%5C%22))%22%7D" target="_blank"><code>b99f6ace</code></a> <em>finished-without-submitting</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22caf86e3e-06f5-462a-911d-45cdb285a7b6%5C%22))%22%7D" target="_blank"><code>caf86e3e</code></a> <em>chunked-planning-no-submit</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%225c8f38fa-a71c-4cf5-81e5-465e8789001d%5C%22))%22%7D" target="_blank"><code>5c8f38fa</code></a> <em>sdk-spelunking-no-edit</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%226cbefec4-7453-461d-815c-6d06c022646d%5C%22))%22%7D" target="_blank"><code>6cbefec4</code></a> <em>fragmented-retry-no-submit</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22c5845fa5-cd06-4889-ad3b-252c0ba9db66%5C%22))%22%7D" target="_blank"><code>c5845fa5</code></a> <em>missing-submit-and-credential-placeholder-misuse</em>

### Sandbox/Infrastructure Outage (Expired Sandbox Token, 502, Bad Gateway) &nbsp;<small>(6 threads)</small>

Failures caused by external infrastructure: Daytona sandbox session tokens dying so every execute_command/write_file returns 'Expired token'; Cloudflare 502/origin_bad_gateway from the orchestrator API; 'failed to get runner info' on sandbox provisioning; Service Unavailable. No code defect — agents often loop retrying instead of escalating.

**Pattern tags:** `upstream-bad-gateway`, `expired-sandbox-token`, `expired-sandbox-token-loop`, `phantom-waf-block-loop`, `sandbox-network-debug-thrash`, `restricted-ip-localhost-daemon`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%222ffe8ec3-c554-4496-9bb6-fb2a5dec0b97%5C%22))%22%7D" target="_blank"><code>2ffe8ec3</code></a> <em>upstream-bad-gateway</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%225ea44f8a-15ea-45b3-b5d0-f8ea9470a07f%5C%22))%22%7D" target="_blank"><code>5ea44f8a</code></a> <em>expired-sandbox-token</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22494c90d0-a1a7-4456-9a6b-bf621ebaffc1%5C%22))%22%7D" target="_blank"><code>494c90d0</code></a> <em>expired-sandbox-token</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%222b8a0aeb-7b70-4b2f-8177-5215bac92f1a%5C%22))%22%7D" target="_blank"><code>2b8a0aeb</code></a> <em>expired-sandbox-token-loop</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%228a23e385-1165-47ea-8707-a54287a7bbb6%5C%22))%22%7D" target="_blank"><code>8a23e385</code></a> <em>phantom-waf-block-loop</em>

### Verify-Built-Workflow Failures Left Unresolved &nbsp;<small>(6 threads)</small>

submit-workflow succeeds but verify-built-workflow / executions.run reveals real runtime issues (missing credentials, unresolved RLC documentId/sheetName placeholders, malformed Code node return shape, restricted IP/localhost calls, MySQL connection refused, IMAP timeout, OpenAI quota) that the agent dismisses as 'expected' or fails to fix, leaving a saved-but-non-executing workflow.

**Pattern tags:** `imap-trigger-verify-timeout`, `restricted-ip-localhost-daemon`, `credential-access-and-data-shape-loop`, `submit-fix-loop-no-credentials`, `deferred-credentials-submit-failure`, `missing-credentials-mock-discovery`

**Examples:** <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22c043d3a5-efd6-44d6-92ff-2026579830aa%5C%22))%22%7D" target="_blank"><code>c043d3a5</code></a> <em>imap-trigger-verify-timeout</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22457df95b-9fc9-4f9f-a95b-c4ddbe0c1282%5C%22))%22%7D" target="_blank"><code>457df95b</code></a> <em>credential-access-and-data-shape-loop</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%223f221af9-5707-4dce-a5b6-b0918565e44a%5C%22))%22%7D" target="_blank"><code>3f221af9</code></a> <em>submit-fix-loop-no-credentials</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%226b4ef4f8-a998-42c6-97f0-c1ee7407011c%5C%22))%22%7D" target="_blank"><code>6b4ef4f8</code></a> <em>missing-credentials-mock-discovery</em> &nbsp;·&nbsp; <a href="https://smith.langchain.com/o/a7b41b9d-bee0-4cf0-9786-f4600380f803/projects/p/cbeb4b95-5d9c-40b4-81f5-6239a427d632?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22b1d73388-da63-482c-aea1-01863a916e50%5C%22))%22%7D" target="_blank"><code>b1d73388</code></a> <em>restricted-ip-localhost-daemon</em>

### All sampled narratives

Sortable table of every thread we sent through the LLM.


In [10]:
rows = []
for n in nb_blob['narratives']:
    nv = n.get('narrative') or {}
    rows.append({
        'thread': f'<a href="{thread_url(n["thread_id"])}" target="_blank">{n["thread_id"][:8]}</a>',
        'tag': nv.get('pattern_tag', '?'),
        'codes': ', '.join(n.get('violation_codes', [])),
        'intent': nv.get('user_intent', ''),
        'failure': nv.get('failure_narrative', ''),
        'conf': nv.get('confidence', ''),
    })
df_n = pd.DataFrame(rows)
display(HTML(df_n.to_html(escape=False, index=False, max_rows=None)))


thread,tag,codes,intent,failure,conf
0e0bcab0,repeated-full-file-rewrites,write_thrash,Build a scheduled workflow that pulls AI meeting notes from a Notion database and writes each page as a Markdown file to a local folder.,"The builder rewrote the entire src/workflow.ts file five times in a row (calls 6–10) instead of using edit_file for incremental changes, churning through nearly identical 8–9KB versions before ever running tsc. When tsc finally ran it surfaced TS2322 errors because placeholder() was passed where a CredentialReference was required; the agent ""fixed"" this by hardcoding an unrelated existing credential id ('cFujd9v7PjvzWMKC' / ""Product Feedback Bot Token"") into the Notion node, contradicting the task's instruction to use placeholder() for the Notion credential. verify-built-workflow then reported the workflow has issues and cannot execute, and the user denied the resume prompt.",high
23d500c2,credential-setup-suspend-timeout,"builder_retry_in_thread, hard_failure","Build a French-language conversational AI agent workflow with Chat Trigger, Anthropic Claude LLM, Window Buffer Memory, and tools (Wikipedia, Calculator, Google Sheets, Gmail), using mocked credentials if real ones aren't available.","The user's instance had zero credentials (`credentials list` returned empty), and despite the task explicitly saying ""credentials mockés si non disponibles"", the builder did not stub/mock them — it submitted the workflow which failed with ""You don't have access to the credentials in the 'Claude Sonnet' node"". The builder then triggered a `credentials setup` request that suspended the agent waiting for human credential input, but the suspension token expired before the user responded, returning ""Expired token"" as the final error. This pattern repeated identically across all three builder retries (full toolset, reduced toolset, minimal single-tool), each suspending on the anthropicApi credential request and timing out.",high
783a1ed1,mock-credential-id-rejected,"builder_retry_in_thread, hard_failure, token_outlier",Build a daily-scheduled AI agent that uses web search to find LinkedIn/Facebook real-estate posts in the NCR region and drafts comments saved to Notion.,"Builder 1 hit an ""Expired token"" auth error mid-run right after the credentials setup wizard suspended for SerpAPI/Anthropic/Notion, killing the first attempt. Builder 2 succeeded only after two submit-workflow rejections — first for a missing '=' prefix on a {{ $now }} expression and a databaseId object instead of string ([MISSING_EXPRESSION_PREFIX] / [INVALID_PARAMETER]), then a second TS1005/TS1135 syntax error introduced when wrapping the prompt in expr(), then a credential-access error that required running the notionApi credential setup flow. Subsequent builders (4 and 7) repeatedly used a fake 'MOCKED'/'MOCK_GROQ_CREDENTIAL' id which submit-workflow rejected with ""You don't have access to the credentials"", forcing a credential setup round-trip each time, and post-build verify-built-workflow runs kept failing on the Google Search (toolSerpApi) sub-node despite the orchestrator's task being unrelated to it — leaving the final workflow saved but non-executing.",high
055fefda,mock-credential-id-rejected,"builder_retry_in_thread, hard_failure","Build a daily NVD CVE alert workflow that emails high/critical vulnerabilities, then swap the Outlook email node for a Gmail node.","Builder 1 succeeded only after stripping the credentials block from the Outlook node so the placeholder/mock id wouldn't trigger the ""you don't have access to the credentials"" error from submit-workflow. Builders 2, 3, and 4 ignored that lesson: each replaced the node with a Gmail node still hard-coding `credentials: { gmailOAuth2: { id: 'mock-gmail-oauth2', name: 'Gmail OAuth2' } }`, so submit-workflow rejected the save with the same ""You don't have access to the credentials in the 'Send CVE Alert Email' node"" error. They then fired the `credentials` setup tool, whic